# 08 · Avaliação retrospectiva da correção de intensidade carbônica

Experimento de Thiago Menezes. Avalia a transferência de F, calibrado no 07 com dados históricos e cortes localizados com CO₂ observado de 2025, às previsões anuais A–D já produzidas pelo 06. M0 conserva a previsão original, M1 aplica F a partir do primeiro corte, e M2 aplica F e depois F². Nenhum estimador é reajustado.

Execute o 07 primeiro. Por padrão o 08 lê as 6.249 previsões de 2025 do 06 preservadas em `reproducibility/iatan/06_anual_2025/`. A rodada anual arquivada tem ID `20261119T013458Z_963fc189`; o identificador citado na cópia original de Thiago é distinto. Suas métricas de 2025 foram reproduzidas na precisão publicada. Para avaliar outra rodada anual concluída do 06, informe o identificador em `IATAN_RUN06_ANUAL`.

A amostra do 06 inclui 6.249 horas em operação; 6.002 horas satisfazem também os critérios do 07, incluindo geração positiva. A aplicação do fator depende da posição temporal: previsões anteriores ao primeiro corte são mantidas. O notebook grava em `outputs/notebook_08_correcao_carbonica/` previsões, métricas, períodos, bootstrap e sensibilidade ao comprimento do bloco. As próximas execuções recalculam esses arquivos.

As entradas do XGBoost no 06 são cumulativas. A usa geração corrente. B acrescenta o histórico de geração (defasagens e estatísticas móveis), C acrescenta os estados HMM a B e D acrescenta os atributos do DFG a C.

## Mapa de fundamentação científica do experimento

| Parte do código | Base bibliográfica | Qual decisão é própria deste estudo? |
|---|---|---|
| Importação e auditoria dos artefatos (01–03) | EPA (2022, p. 8, *Hourly Data*); Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3) | Harmonizar previsões e medidas pelas horas coincidentes, verificando divergências |
| Origem dos modelos A–D | Chen e Guestrin (2016, §§2.1–2.2, p. 786); Rabiner (1989, §§II.B–III.A, pp. 260–263); van der Aalst (2022, pp. 10 e 12) | A–D foram treinados no Notebook 06; **não são estimados aqui** |
| Correção dos modelos com F (04–05) | Gama et al. (2014, pp. 44:2 e 44:4); Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11); **Notebook 07** | Transportar um único fator histórico e aplicá-lo cumulativamente, sem novo treino |
| Comparação global, por período e mensal (06–07) | Hyndman e Koehler (2006, §2.1, p. 682); Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3) | Avaliar M0/M1/M2 em amostras horárias coincidentes |
| Bootstrap pareado por blocos (08 e diagnóstico complementar) | Künsch (1989, pp. 1217–1218) | Blocos calendários 7 dias; análise de sensibilidade 3/7/14/21 dias |
| Cautelas de seleção e validade | Jewell et al. (2022, §1, pp. 1082–1084); Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3) | Datas selecionadas com informação de 2025; **ganhos condicionais e retrospectivos** |

**Hipótese testada:** se \(\widehat y_{m,t}^{M0}\) é a previsão já existente do modelo \(m\), as versões corrigidas são \(\widehat y_{m,t}^{M1}=\widehat y_{m,t}^{M0}F^{\mathbf{1}(t\ge\tau_1)}\) e \(\widehat y_{m,t}^{M2}=\widehat y_{m,t}^{M0}F^{\mathbf{1}(t\ge\tau_1)+\mathbf{1}(t\ge\tau_2)}\). A aplicação do **mesmo F** às quatro arquiteturas e sua repetição após dois eventos são **hipóteses deste estudo**, e não regras garantidas pelas referências.

O que as fontes **não demonstram**: que as datas correspondam a manutenções reais, que o efeito seja causal, que o fator permaneça igual após cada mudança ou que esses ganhos possam ser alcançados em tempo real sem observar as emissões futuras.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

if os.environ.get("IATAN_PROJECT_ROOT"):
    root = Path(os.environ["IATAN_PROJECT_ROOT"]).expanduser()
elif drive is not None:
    delivery = Path(os.environ.get("IATAN_DELIVERY_ROOT",
        "/content/drive/MyDrive/Ciência de Dados - Iatan - Process Mining e Experimentos/Iatan 2025 - entrega revisada"))
    root = delivery / "execucao"
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "configs/iatan_inputs.json").is_file()), Path.cwd())
if not all((root / name).is_file() for name in ("iatan/colab.py", "iatan/carbon_tracking.py", "requirements-iatan.txt")):
    raise FileNotFoundError("Execute a versão atual do notebook 00 ou indique sua raiz em IATAN_PROJECT_ROOT.")
os.chdir(root)
sys.path.insert(0, str(Path.cwd()))
PROJECT_ROOT = Path(".")
from iatan.colab import ensure_dependencies
ensure_dependencies(PROJECT_ROOT, install=drive is not None)

import numpy as np
import pandas as pd
from IPython.display import display
ROOT08 = PROJECT_ROOT
run_id = os.environ.get("IATAN_RUN06_ANUAL", "").strip()
if run_id:
    if Path(run_id).name != run_id or run_id in {".", ".."} or "/" in run_id or "\\" in run_id:
        raise ValueError("Informe somente o nome da pasta da rodada anual do 06.")
    RUN06 = ROOT08 / "outputs/phase_05_multiyear" / run_id
    ARQUIVO06 = RUN06 / "resultados/previsoes_pareadas.csv"
    ARQUIVO_METRICAS06 = RUN06 / "resultados/metricas.csv"
    registro06 = json.loads((RUN06 / "resultados/resumo.json").read_text(encoding="utf-8"))
    if registro06.get("status") != "FINISHED":
        raise ValueError("A rodada anual selecionada ainda não está concluída.")
    PAI_MLFLOW06 = registro06["mlflow_run_id"]
else:
    RUN06 = ROOT08 / "reproducibility/iatan/06_anual_2025"
    ARQUIVO06 = RUN06 / "previsoes_pareadas.csv.gz"
    ARQUIVO_METRICAS06 = RUN06 / "metricas.csv"
    PAI_MLFLOW06 = "d3b4efcf936a470c8f520717748ca38d"
PASTA07_FONTE = ROOT08 / "outputs/07_intensidade_carbonica"
ARQUIVO07 = PASTA07_FONTE / "dados/22C_previsoes_2025.parquet"
ARQUIVO_FATOR07 = PASTA07_FONTE / "tabelas/22C_fator_historico.csv"
ARQUIVO_DATAS07 = PASTA07_FONTE / "tabelas/19C_reconciliacao_datas_2025.csv"
ARQUIVO_RELATORIO07 = PASTA07_FONTE / "relatorios/21_resumo_reprodutibilidade.json"
for entrada in (ARQUIVO06, ARQUIVO_METRICAS06, ARQUIVO07, ARQUIVO_FATOR07, ARQUIVO_DATAS07, ARQUIVO_RELATORIO07):
    if not entrada.is_file():
        raise FileNotFoundError(f"Insumo ausente: {entrada}. Confira a sequência 06 → 07 → 08.")
print("Previsões do 06:", ARQUIVO06)
print("Saídas do 07:", PASTA07_FONTE)
# 7. Verificar conteúdo, parâmetros e correção dimensional do Notebook 07.
prev07 = pd.read_parquet(ARQUIVO07)
colunas07_obrigatorias = [
    "timestamp", "tempo_operacao_h", "carga_bruta_mw", "geracao_mwh",
    "co2_t", "pred_M0", "pred_M1", "pred_M2", "n_intervencoes"
]
faltantes07 = [x for x in colunas07_obrigatorias if x not in prev07.columns]
if faltantes07:
    raise ValueError(f"Colunas necessárias ausentes no parquet 07: {faltantes07}")

prev07["timestamp"] = pd.to_datetime(prev07["timestamp"], errors="raise")
if prev07["timestamp"].isna().any() or prev07["timestamp"].duplicated().any():
    raise AssertionError("Timestamps do 07 ausentes ou duplicados.")

# Conferir geração horária (MW × h) e potência (MW) confrontando os campos de origem.
colunas_numericas = [x for x in colunas07_obrigatorias if x != "timestamp"]
for coluna in colunas_numericas:
    prev07[coluna] = pd.to_numeric(prev07[coluna], errors="raise")

if not np.isfinite(prev07[colunas_numericas].to_numpy(dtype=float)).all():
    raise AssertionError("Valores não finitos no parquet 22C do 07.")

mask_operacao_parcial = (
    prev07["tempo_operacao_h"].gt(0)
    & prev07["tempo_operacao_h"].lt(1)
    & prev07["geracao_mwh"].gt(0)
)
if not mask_operacao_parcial.any():
    raise AssertionError(
        "O parquet 22C não contém horas parciais positivas para checar "
        "a origem da geração; revise a população selecionada."
    )

# A base CEMS original foi salva pela etapa de aquisição do Notebook 07.
ARQUIVO_RAW07 = (
    PASTA07_FONTE / "dados" / "iatan2_hourly_raw_2011_2025.parquet"
)
if not ARQUIVO_RAW07.is_file():
    raise FileNotFoundError(
        "Não encontrei a base original CEMS do Notebook 07 para a "
        "auditoria independente de grossLoad: " + str(ARQUIVO_RAW07)
    )

raw07 = pd.read_parquet(
    ARQUIVO_RAW07, columns=["timestamp", "grossLoad", "opTime"]
)
raw07["timestamp"] = pd.to_datetime(raw07["timestamp"], errors="raise")
raw07 = raw07.loc[
    raw07["timestamp"].isin(prev07["timestamp"])
].copy()

if raw07["timestamp"].duplicated().any():
    raise AssertionError("A base CEMS original contém timestamps duplicados no período de comparação.")

auditoria_cems07 = prev07[[
    "timestamp", "tempo_operacao_h", "carga_bruta_mw", "geracao_mwh"
]].merge(
    raw07,
    how="left",
    on="timestamp",
    validate="one_to_one",
    indicator=True
)
if not auditoria_cems07["_merge"].eq("both").all():
    raise AssertionError("Há horas do parquet 22C não encontradas no CEMS original.")

for coluna in ["grossLoad", "opTime"]:
    auditoria_cems07[coluna] = pd.to_numeric(
        auditoria_cems07[coluna], errors="raise"
    )
if not np.isfinite(
    auditoria_cems07[["grossLoad", "opTime"]].to_numpy(dtype=float)
).all():
    raise AssertionError("O CEMS original contém valores não finitos nas horas avaliadas.")

esperado_mwh = auditoria_cems07["grossLoad"] * auditoria_cems07["opTime"]
maior_diferenca_geracao07 = float((auditoria_cems07["geracao_mwh"] - esperado_mwh).abs().max())
if not (np.allclose(auditoria_cems07["geracao_mwh"], esperado_mwh, atol=1e-7, rtol=0)
        and np.allclose(auditoria_cems07["carga_bruta_mw"], auditoria_cems07["grossLoad"], atol=1e-7, rtol=0)):
    raise AssertionError("Energia ou potência divergente dos campos horários do 07.")

# opTime também deve manter os valores originais da EPA.
if not np.allclose(
    auditoria_cems07["tempo_operacao_h"], auditoria_cems07["opTime"],
    rtol=0, atol=1e-7
):
    raise AssertionError("O opTime do 22C diverge do CEMS original.")

# Identidade dimensional adicional: MWh = MW_operacional × opTime.
# Não utilizar esta identidade isoladamente para provar a correção:
# ela também seria satisfeita pelo pipeline antigo que duplicava opTime.
maior_diferenca_identidade07 = float(np.max(np.abs(
    auditoria_cems07["geracao_mwh"].to_numpy(dtype=float)
    - (
        auditoria_cems07["carga_bruta_mw"].to_numpy(dtype=float)
        * auditoria_cems07["tempo_operacao_h"].to_numpy(dtype=float)
    )
)))
if not np.allclose(
    auditoria_cems07["geracao_mwh"],
    auditoria_cems07["carga_bruta_mw"] * auditoria_cems07["tempo_operacao_h"],
    rtol=0, atol=1e-5
):
    raise AssertionError(
        "Inconsistência entre a carga operacional e a geração (MWh = MW × opTime)."
    )

_tabela_fator07 = pd.read_csv(ARQUIVO_FATOR07, float_precision="round_trip")
if len(_tabela_fator07) != 1 or "fator_aprendido" not in _tabela_fator07:
    raise AssertionError("Estrutura inesperada no CSV do fator do Notebook 07.")
FATOR07_CSV = float(_tabela_fator07.loc[0, "fator_aprendido"])
if not np.isfinite(FATOR07_CSV) or FATOR07_CSV <= 0:
    raise AssertionError("Fator histórico inválido no CSV do 07.")

with ARQUIVO_RELATORIO07.open("r", encoding="utf-8") as f:
    RELATORIO07 = json.load(f)

FATOR07_RELATORIO = float(RELATORIO07["fator_historico_principal"])
if FATOR07_CSV != FATOR07_RELATORIO:
    raise AssertionError("Fator CSV e fator do relatório 21 do Notebook 07 divergem.")

_rmse07_m0 = float(np.sqrt(np.mean((prev07["pred_M0"] - prev07["co2_t"])**2)))
_rmse07_m2 = float(np.sqrt(np.mean((prev07["pred_M2"] - prev07["co2_t"])**2)))
for nome, calculado, registrado in [
    ("RMSE M0", _rmse07_m0, float(RELATORIO07["rmse_m0"])),
    ("RMSE M2", _rmse07_m2, float(RELATORIO07["rmse_m2"])),
]:
    if not np.isclose(calculado, registrado, rtol=0, atol=1e-5):
        raise AssertionError(f"{nome} do parquet diverge do relatório final do 07.")

print("\nAUDITORIA DOS ARTEFATOS 07:")
print("Linhas:", len(prev07))
print("Período:", prev07["timestamp"].min(), "até", prev07["timestamp"].max())
print("Horas parciais auditadas:", int(mask_operacao_parcial.sum()))
print("Diferença máxima: geração MWh vs grossLoad × opTime:", maior_diferenca_geracao07)
print("Diferença máxima: geração MWh vs carga × opTime:", maior_diferenca_identidade07)
print(f"Fator conferido (CSV + relatório): {FATOR07_CSV:.10f}")
print(f"RMSE 07 (M0/M2): {_rmse07_m0:.4f} / {_rmse07_m2:.4f} tCO₂")

# 8. Status
print("\n" + "=" * 65)
print("AUDITORIA INICIAL CONCLUÍDA")
print("Nenhum modelo foi treinado.")
print("Nenhum arquivo original foi modificado.")
print("=" * 65)

## Correspondência entre os dados do 06 e do 07

Confere timestamps, potência bruta, duração da operação, massa e previsões. O 07 confronta a energia horária com o agregado anual informado pela EPA. O 08 compara de novo o valor linha a linha nas horas compartilhadas e verifica as métricas contra a tabela da rodada anual do 06.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 02 — IMPORTAÇÃO E AUDITORIA DAS PREVISÕES
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

# 1. Verificar os resultados da Célula 01
if "RUN06" not in globals() or RUN06 is None:
    raise RuntimeError("Execute a Célula 01 e selecione a rodada 06.")

if "ARQUIVO07" not in globals() or ARQUIVO07 is None:
    raise RuntimeError("Arquivo 22C do Notebook 07 não selecionado.")



if not ARQUIVO06.is_file():
    raise FileNotFoundError(ARQUIVO06)

# 2. Carregar os dois conjuntos
prev06_08 = pd.read_csv(ARQUIVO06, low_memory=False)
prev07_08 = pd.read_parquet(ARQUIVO07)

print("=" * 65)
print("NOTEBOOK 08 — AUDITORIA DAS PREVISÕES")
print("=" * 65)

# 3. Examinar a estrutura original
for nome, df in [
    ("Notebook 06", prev06_08),
    ("Notebook 07", prev07_08),
]:
    print(f"\n{nome}")
    print("Dimensões:", df.shape)
    print("Colunas:")
    print(df.columns.tolist())
    print("\nPrimeiras observações:")
    display(df.head(3))

# 4. Identificar a coluna temporal do Notebook 06
candidatas_tempo = [
    "timestamp", "datetime", "data_hora",
    "date_time", "time", "Date"
]

coluna_tempo06 = next(
    (c for c in candidatas_tempo if c in prev06_08.columns),
    None
)

if coluna_tempo06 is None:
    print("\nATENÇÃO: coluna temporal do Notebook 06")
    print("não identificada automaticamente.")
    print("Será necessário verificar o esquema acima.")
else:
    prev06_08["timestamp_auditoria"] = pd.to_datetime(
        prev06_08[coluna_tempo06], errors="coerce"
    )

# 5. Padronizar timestamp do Notebook 07
prev07_08["timestamp_auditoria"] = pd.to_datetime(
    prev07_08["timestamp"], errors="coerce"
)

# 6. Examinar colunas relacionadas às previsões
termos = [
    "pred", "prev", "modelo", "model",
    "observ", "real", "co2", "y_true", "y_pred"
]

print("\nCOLUNAS CANDIDATAS — NOTEBOOK 06")
print([
    c for c in prev06_08.columns
    if any(t in c.lower() for t in termos)
])

# 7. Auditar período e timestamps
if "timestamp_auditoria" in prev06_08.columns:

    dados06_2025 = prev06_08.loc[
        prev06_08["timestamp_auditoria"].dt.year.eq(2025)
    ].copy()

    tempos06 = pd.Index(
        dados06_2025["timestamp_auditoria"].dropna().unique()
    )

    tempos07 = pd.Index(
        prev07_08["timestamp_auditoria"].dropna().unique()
    )

    comuns = tempos06.intersection(tempos07)

    auditoria = pd.DataFrame([{
        "registros_06_2025": len(dados06_2025),
        "horas_unicas_06": len(tempos06),
        "registros_07": len(prev07_08),
        "horas_unicas_07": len(tempos07),
        "horas_comuns": len(comuns),
        "horas_exclusivas_06": len(tempos06.difference(tempos07)),
        "horas_exclusivas_07": len(tempos07.difference(tempos06)),
    }])

    print("\nCOBERTURA TEMPORAL")
    display(auditoria)

    print("\nPeríodo Notebook 06 (2025):")
    print(dados06_2025["timestamp_auditoria"].min())
    print(dados06_2025["timestamp_auditoria"].max())

    print("\nPeríodo Notebook 07:")
    print(prev07_08["timestamp_auditoria"].min())
    print(prev07_08["timestamp_auditoria"].max())

    print("\nDuplicações de timestamp:")
    print(
        "Notebook 06:",
        dados06_2025["timestamp_auditoria"].duplicated().sum()
    )
    print(
        "Notebook 07:",
        prev07_08["timestamp_auditoria"].duplicated().sum()
    )

# 8. Verificar métricas originais do Notebook 06


if ARQUIVO_METRICAS06.is_file():
    metricas06_08 = pd.read_csv(ARQUIVO_METRICAS06)

    print("\nMÉTRICAS ORIGINAIS DO NOTEBOOK 06")
    display(metricas06_08.tail(12))

# 9. Encerramento
print("\n" + "=" * 65)
print("CÉLULA 02 CONCLUÍDA")
print("Nenhuma correção foi aplicada.")
print("Nenhum modelo foi treinado.")
print("Nenhum arquivo original foi modificado.")
print("=" * 65)

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 03 — REPRODUÇÃO DAS MÉTRICAS E VALIDAÇÃO
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# 1. Verificar dependências das células anteriores
necessarias = [
    "prev06_08", "prev07_08", "metricas06_08"
]
ausentes = [v for v in necessarias if v not in globals()]

if ausentes:
    raise RuntimeError(
        f"Execute a Célula 02. Variáveis ausentes: {ausentes}"
    )

MODELOS08 = ["A", "B", "C", "D"]

# 2. Selecionar 2025 e validar dados
base06_08 = prev06_08.loc[
    prev06_08["ano"].astype(str).eq("2025"),
    ["timestamp", "co2_metric_tons", *MODELOS08]
].copy()

base06_08["timestamp"] = pd.to_datetime(
    base06_08["timestamp"], errors="raise"
)

base07_08 = prev07_08[
    ["timestamp", "co2_t", "n_intervencoes"]
].copy()

base07_08["timestamp"] = pd.to_datetime(
    base07_08["timestamp"], errors="raise"
)

for nome, df in [("06", base06_08), ("07", base07_08)]:
    if df["timestamp"].isna().any():
        raise ValueError(f"Timestamp ausente no Notebook {nome}")
    if df["timestamp"].duplicated().any():
        raise ValueError(f"Timestamps duplicados no Notebook {nome}")

for col in ["co2_metric_tons", *MODELOS08]:
    base06_08[col] = pd.to_numeric(
        base06_08[col], errors="raise"
    )
    if not np.isfinite(base06_08[col].to_numpy()).all():
        raise ValueError(f"Valores inválidos na coluna {col}")

# 3. Função única para as métricas
def calcular_metricas08(real, previsto):
    real = np.asarray(real, dtype=float)
    previsto = np.asarray(previsto, dtype=float)

    erro = previsto - real

    return {
        "n": len(real),
        "rmse": float(np.sqrt(np.mean(erro ** 2))),
        "mae": float(np.mean(np.abs(erro))),
        "vies": float(np.mean(erro))
    }

# 4. Reproduzir métricas originais de 2025
linhas = []

for modelo in MODELOS08:
    resultado = calcular_metricas08(
        base06_08["co2_metric_tons"],
        base06_08[modelo]
    )
    linhas.append({"modelo": modelo, **resultado})

metricas_reproduzidas08 = pd.DataFrame(linhas)

referencia08 = metricas06_08.loc[
    metricas06_08["ano"].astype(str).eq("2025"),
    ["cenario", "n", "rmse", "mae", "vies"]
].rename(columns={"cenario": "modelo"})

comparacao08 = metricas_reproduzidas08.merge(
    referencia08,
    on="modelo",
    how="left",
    suffixes=("_calculado", "_original"),
    validate="one_to_one"
)

for metrica in ["rmse", "mae", "vies"]:
    comparacao08[f"delta_{metrica}"] = (
        comparacao08[f"{metrica}_calculado"]
        - comparacao08[f"{metrica}_original"]
    )

comparacao08["reproduzido"] = (
    comparacao08["n_calculado"].eq(
        comparacao08["n_original"]
    )
    & np.isclose(
        comparacao08["rmse_calculado"],
        comparacao08["rmse_original"],
        atol=1e-6, rtol=0
    )
    & np.isclose(
        comparacao08["mae_calculado"],
        comparacao08["mae_original"],
        atol=1e-6, rtol=0
    )
    & np.isclose(
        comparacao08["vies_calculado"],
        comparacao08["vies_original"],
        atol=1e-6, rtol=0
    )
)

print("=" * 65)
print("REPRODUÇÃO DAS MÉTRICAS — NOTEBOOK 06 / 2025")
print("=" * 65)

display(comparacao08.round(6))

if not comparacao08["reproduzido"].all():
    raise AssertionError(
        "As métricas originais não foram reproduzidas. "
        "Investigue antes de aplicar correções."
    )

print("\nTodas as métricas foram reproduzidas.")

# 5. Comparar valores observados nas horas comuns
horas_comuns08 = base06_08.merge(
    base07_08,
    on="timestamp",
    how="inner",
    validate="one_to_one"
)

horas_comuns08["diferenca_co2"] = (
    horas_comuns08["co2_metric_tons"]
    - horas_comuns08["co2_t"]
)

diferencas = horas_comuns08["diferenca_co2"].abs()
if len(horas_comuns08) != len(base07_08) or not np.isfinite(diferencas).all() or diferencas.max() > 1e-7:
    raise ValueError("As horas elegíveis do 07 e as massas observadas devem coincidir com as do 06.")


print("\n" + "=" * 65)
print("CONSISTÊNCIA DOS VALORES OBSERVADOS")
print("=" * 65)

print("Horas do Notebook 06:", len(base06_08))
print("Horas do Notebook 07:", len(base07_08))
print("Horas comuns:", len(horas_comuns08))

print("Diferença absoluta média (t):",
      round(float(diferencas.mean()), 8))

print("Diferença absoluta máxima (t):",
      round(float(diferencas.max()), 8))

print("Horas com diferença > 0,001 t:",
      int((diferencas > 0.001).sum()))

print("\nDistribuição das intervenções nas horas comuns:")
display(
    horas_comuns08["n_intervencoes"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_intervencoes")
    .reset_index(name="horas")
)

print("\n" + "=" * 65)
print("CÉLULA 03 CONCLUÍDA")
print("Nenhuma correção aplicada.")
print("Nenhum modelo treinado.")
print("=" * 65)

## Fator e datas do 07

O 08 lê F com a precisão original do CSV e confere o valor com o relatório JSON. As previsões do 07 permitem verificar separadamente a multiplicação por F ou F². A tabela 19C fornece os cortes de 2025. A calibração, os marcos temporais e a regra de repetição do fator pertencem a este experimento. Gama et al. (2014, pp. 44:2 e 44:4) discutem mudanças temporais na relação entre variáveis; Jewell et al. (2022, §1, pp. 1082–1084) discutem inferência após seleção de cortes.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 04 — REPRODUÇÃO DAS CORREÇÕES DO NOTEBOOK 07
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# 1. Verificar a Célula 03
necessarias = ["base06_08", "base07_08", "prev07_08"]
ausentes = [v for v in necessarias if v not in globals()]

if ausentes:
    raise RuntimeError(
        f"Execute as células anteriores. Ausentes: {ausentes}"
    )

# 2. Datas de intervenção do Experimento 22C
cortes07 = pd.read_csv(ARQUIVO_DATAS07)
INTERVENCOES08 = pd.DatetimeIndex(pd.to_datetime(cortes07["instante_usado_22C_22D"]))
if len(INTERVENCOES08) != 2 or not INTERVENCOES08.is_monotonic_increasing or not INTERVENCOES08.is_unique:
    raise ValueError("São necessários dois cortes cronológicos distintos do 07.")

# 3. Recuperar o fator histórico das previsões do Notebook 07
# O fator foi estimado originalmente com dados históricos.
# Aqui apenas recuperamos seu valor, sem recalibrá-lo.

fonte07 = prev07_08.copy()
fonte07["timestamp"] = pd.to_datetime(fonte07["timestamp"])

colunas = ["pred_M0", "pred_M1", "pred_M2", "n_intervencoes"]

for coluna in colunas:
    if coluna not in fonte07.columns:
        raise KeyError(f"Coluna ausente: {coluna}")

# Recuperar F usando as horas após a primeira intervenção,
# nas quais pred_M1 = pred_M0 * F.
mascara = (
    (fonte07["n_intervencoes"] >= 1)
    & np.isfinite(fonte07["pred_M0"])
    & np.isfinite(fonte07["pred_M1"])
    & (fonte07["pred_M0"].abs() > 1e-10)
)

fatores = (
    fonte07.loc[mascara, "pred_M1"]
    / fonte07.loc[mascara, "pred_M0"]
)

if fatores.empty:
    raise ValueError("Não foi possível recuperar o fator.")

FATOR08 = FATOR07_CSV  # Valor binário completo exportado pelo 07; a razão verifica o parquet.

if not np.isfinite(FATOR08) or FATOR08 <= 0:
    raise ValueError("Fator histórico inválido.")

if not np.allclose(fatores, FATOR08, rtol=1e-9, atol=1e-9):
    raise AssertionError(
        "O fator recuperado não é constante no Notebook 07."
    )

# Confronto com arquivos independentes do Notebook 07 FINAL,
# impedindo reaproveitar fator antigo por engano.
if not np.isclose(FATOR08, FATOR07_CSV, rtol=0, atol=1e-10):
    raise AssertionError(
        "O fator recuperado do parquet não coincide com "
        "22C_fator_historico.csv (Notebook 07 corrigido)."
    )
if not np.isclose(FATOR08, FATOR07_RELATORIO, rtol=0, atol=1e-10):
    raise AssertionError("Fator recuperado diverge do relatório 21 do Notebook 07.")

_cortes_relatorio07 = pd.to_datetime(RELATORIO07["cortes_2025"])
if len(_cortes_relatorio07) != 2 or not np.array_equal(
    _cortes_relatorio07.to_numpy(dtype="datetime64[ns]"),
    INTERVENCOES08.to_numpy(dtype="datetime64[ns]")
):
    raise AssertionError(
        "Datas do Notebook 08 divergem da reconciliação final do Notebook 07."
    )

# 4. Reproduzir a contagem de intervenções
datas_eventos = INTERVENCOES08.to_numpy(dtype="datetime64[ns]")
datas07 = fonte07["timestamp"].to_numpy(dtype="datetime64[ns]")

n_calculado = np.searchsorted(
    datas_eventos, datas07, side="right"
)

if not np.array_equal(
    n_calculado,
    fonte07["n_intervencoes"].to_numpy()
):
    raise AssertionError(
        "As datas não reproduzem as intervenções do Notebook 07."
    )

# 5. Reproduzir as previsões corrigidas do Notebook 07
pred_m1_reproduzida = (
    fonte07["pred_M0"].to_numpy()
    * np.where(n_calculado >= 1, FATOR08, 1.0)
)

pred_m2_reproduzida = (
    fonte07["pred_M0"].to_numpy()
    * np.power(FATOR08, n_calculado)
)

erro_m1 = np.max(np.abs(
    pred_m1_reproduzida - fonte07["pred_M1"].to_numpy()
))

erro_m2 = np.max(np.abs(
    pred_m2_reproduzida - fonte07["pred_M2"].to_numpy()
))

if not np.allclose(
    pred_m1_reproduzida,
    fonte07["pred_M1"],
    rtol=1e-9, atol=1e-7
):
    raise AssertionError("Não foi possível reproduzir M1.")

if not np.allclose(
    pred_m2_reproduzida,
    fonte07["pred_M2"],
    rtol=1e-9, atol=1e-7
):
    raise AssertionError("Não foi possível reproduzir M2.")

# 6. Exibir os parâmetros validados
print("=" * 65)
print("NOTEBOOK 08 — VALIDAÇÃO DAS CORREÇÕES")
print("=" * 65)

print(f"\nFator histórico recuperado: {FATOR08:.10f}")
print(f"Redução após março: {(1-FATOR08)*100:.4f}%")
print(f"Redução cumulativa: {(1-FATOR08**2)*100:.4f}%")

print("\nDatas:")
for data in INTERVENCOES08:
    print(" -", data)

print("\nErros máximos de reprodução:")
print(f"M1: {erro_m1:.12f} t")
print(f"M2: {erro_m2:.12f} t")

tabela_correcoes08 = pd.DataFrame({
    "periodo": [
        "Antes de março",
        "Março a julho",
        "Após julho"
    ],
    "n_intervencoes": [0, 1, 2],
    "multiplicador_M0": [1.0, 1.0, 1.0],
    "multiplicador_M1": [1.0, FATOR08, FATOR08],
    "multiplicador_M2": [1.0, FATOR08, FATOR08**2]
})

display(tabela_correcoes08.round(8))

print("\nCORREÇÕES DO NOTEBOOK 07 REPRODUZIDAS.")
print("Nenhuma previsão dos modelos A–D foi alterada.")

## Pós-processamento multiplicativo sem retreinamento — Célula 05

A adaptação de previsões a mudanças de regime é motivada por **Gama et al. (2014, pp. 44:2 e 44:4)** e, no contexto termelétrico, por **Akpan e Fuls (2021, §3.3, pp. 8–9, figs. 10–11)**. **Não se afirma que tais trabalhos prescrevem a regra implementada.** A aplicação de \(\widehat y_{m,t}^{M0}F^k\), com \(k=0,1,2\), é uma **hipótese operacional própria**, formulada no Notebook 07 e testada aqui nos quatro modelos A–D já treinados. As mesmas datas e o mesmo fator são usados em todos os modelos para tornar o efeito da correção comparável dentro desta amostra.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 05 — APLICAÇÃO DAS CORREÇÕES AOS MODELOS A–D
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# 1. Verificar resultados anteriores
necessarias = [
    "base06_08",
    "base07_08",
    "FATOR08",
    "INTERVENCOES08",
    "MODELOS08"
]

ausentes = [v for v in necessarias if v not in globals()]

if ausentes:
    raise RuntimeError(
        f"Execute as células anteriores. Ausentes: {ausentes}"
    )

# 2. Preservar as previsões originais
previsoes08 = base06_08.copy()

previsoes08 = previsoes08.sort_values(
    "timestamp"
).reset_index(drop=True)

if len(previsoes08) != 6249:
    raise AssertionError(
        "Quantidade de horas diferente da referência."
    )

# 3. Identificar intervenções por timestamp
datas_eventos = INTERVENCOES08.to_numpy(
    dtype="datetime64[ns]"
)

datas_horas = previsoes08["timestamp"].to_numpy(
    dtype="datetime64[ns]"
)

previsoes08["n_intervencoes"] = np.searchsorted(
    datas_eventos,
    datas_horas,
    side="right"
)

# 4. Construir multiplicadores
previsoes08["fator_M0"] = 1.0

previsoes08["fator_M1"] = np.where(
    previsoes08["n_intervencoes"] >= 1,
    FATOR08,
    1.0
)

previsoes08["fator_M2"] = np.power(
    FATOR08,
    previsoes08["n_intervencoes"]
)

# 5. Aplicar correções a cada modelo
for modelo in MODELOS08:

    previsoes08[f"{modelo}_M0"] = previsoes08[modelo]

    previsoes08[f"{modelo}_M1"] = (
        previsoes08[modelo]
        * previsoes08["fator_M1"]
    )

    previsoes08[f"{modelo}_M2"] = (
        previsoes08[modelo]
        * previsoes08["fator_M2"]
    )

# 6. Identificar horas comuns aos dois notebooks
tempos07 = pd.Index(base07_08["timestamp"])

previsoes08["hora_comum_07"] = (
    previsoes08["timestamp"].isin(tempos07)
)

# 7. Validar integridade das previsões
for modelo in MODELOS08:

    original = previsoes08[modelo].to_numpy()
    m0 = previsoes08[f"{modelo}_M0"].to_numpy()
    m1 = previsoes08[f"{modelo}_M1"].to_numpy()
    m2 = previsoes08[f"{modelo}_M2"].to_numpy()

    if not np.array_equal(original, m0):
        raise AssertionError(
            f"Previsão original alterada: {modelo}"
        )

    for nome, valores in [
        ("M1", m1),
        ("M2", m2)
    ]:
        if not np.isfinite(valores).all():
            raise AssertionError(
                f"Valores inválidos em {modelo}_{nome}"
            )

# 8. Validar contagem de intervenções nas horas comuns
comparacao_eventos08 = previsoes08.loc[
    previsoes08["hora_comum_07"],
    ["timestamp", "n_intervencoes"]
].merge(
    base07_08[["timestamp", "n_intervencoes"]],
    on="timestamp",
    suffixes=("_08", "_07"),
    validate="one_to_one"
)

if not (
    comparacao_eventos08["n_intervencoes_08"]
    == comparacao_eventos08["n_intervencoes_07"]
).all():
    raise AssertionError(
        "Contagem de intervenções diverge do Notebook 07."
    )

# 9. Resumo do conjunto produzido
resumo05_08 = (
    previsoes08
    .groupby("n_intervencoes")
    .agg(
        horas=("timestamp", "size"),
        horas_comuns_07=("hora_comum_07", "sum"),
        fator_M1=("fator_M1", "first"),
        fator_M2=("fator_M2", "first")
    )
    .reset_index()
)

print("=" * 65)
print("NOTEBOOK 08 — PREVISÕES CORRIGIDAS")
print("=" * 65)

print("\nFator histórico:", round(FATOR08, 10))
print("Total de horas:", len(previsoes08))
print("Horas comuns ao Notebook 07:",
      int(previsoes08["hora_comum_07"].sum()))

print("\nDistribuição por intervenção:")
display(resumo05_08.round(8))

colunas_amostra = [
    "timestamp",
    "co2_metric_tons",
    "n_intervencoes",
    "A_M0", "A_M1", "A_M2",
    "D_M0", "D_M1", "D_M2"
]

print("\nAmostra das previsões:")
display(previsoes08[colunas_amostra].head(5))

print("\nModelos:", MODELOS08)
print("Cenários: M0, M1, M2")
print("Total de séries preditivas:", len(MODELOS08) * 3)

print("\nCÉLULA 05 CONCLUÍDA")
print("Nenhum modelo foi retreinado.")
print("Arquivos originais preservados.")

## Métricas e comparação global — Células 06–07

**Hyndman e Koehler (2006, §2.1, p. 682)** fundamentam a discussão sobre medidas de erro. São usados **RMSE** (penaliza mais intensamente grandes erros), **MAE** (erro absoluto médio) e **viés** (média de previsão menos observado). A análise é **pareada** no sentido de que M0, M1 e M2 do mesmo modelo usam as mesmas horas observadas. **Cerqueira et al. (2020, pp. 1999–2000, figs. 1–3)** orientam a cautela na extrapolação temporal: ganhos calculados após conhecer as datas retrospectivas não equivalem ao desempenho que seria obtido se a informação tivesse de ser descoberta antecipadamente.

Para o viés o estudo usa previsto − observado; Hyndman e Koehler usam a convenção oposta para erro assinado. RMSE e MAE têm a mesma definição nas duas convenções.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 06 — AVALIAÇÃO GLOBAL DAS CORREÇÕES
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# 1. Verificar a Célula 05
if "previsoes08" not in globals():
    raise RuntimeError("Execute a Célula 05 primeiro.")

MODELOS08 = ["A", "B", "C", "D"]
CENARIOS08 = ["M0", "M1", "M2"]

# 2. Função de avaliação
def avaliar08(df, modelo, cenario):
    real = df["co2_metric_tons"].to_numpy(dtype=float)
    previsto = df[f"{modelo}_{cenario}"].to_numpy(dtype=float)

    erro = previsto - real

    return {
        "modelo": modelo,
        "cenario": cenario,
        "n": len(df),
        "rmse": np.sqrt(np.mean(erro**2)),
        "mae": np.mean(np.abs(erro)),
        "vies": np.mean(erro)
    }

# 3. Avaliar duas populações
populacoes08 = {
    "principal_6249h": previsoes08,
    "comum_6002h": previsoes08.loc[
        previsoes08["hora_comum_07"]
    ].copy()
}

resultados = []

for populacao, dados in populacoes08.items():
    for modelo in MODELOS08:
        for cenario in CENARIOS08:
            linha = avaliar08(dados, modelo, cenario)
            linha["populacao"] = populacao
            resultados.append(linha)

metricas_globais08 = pd.DataFrame(resultados)

# 4. Calcular ganhos em relação ao cenário original
referencias = (
    metricas_globais08
    .loc[
        metricas_globais08["cenario"].eq("M0"),
        ["populacao", "modelo", "rmse", "mae", "vies"]
    ]
    .rename(columns={
        "rmse": "rmse_M0",
        "mae": "mae_M0",
        "vies": "vies_M0"
    })
)

metricas_globais08 = metricas_globais08.merge(
    referencias,
    on=["populacao", "modelo"],
    how="left",
    validate="many_to_one"
)

metricas_globais08["ganho_rmse_pct"] = (
    100 * (
        metricas_globais08["rmse_M0"]
        - metricas_globais08["rmse"]
    ) / metricas_globais08["rmse_M0"]
)

metricas_globais08["ganho_mae_pct"] = (
    100 * (
        metricas_globais08["mae_M0"]
        - metricas_globais08["mae"]
    ) / metricas_globais08["mae_M0"]
)

# Redução do viés absoluto, pois o viés pode ser negativo.
metricas_globais08["ganho_vies_abs_pct"] = (
    100 * (
        np.abs(metricas_globais08["vies_M0"])
        - np.abs(metricas_globais08["vies"])
    ) / np.abs(metricas_globais08["vies_M0"])
)

# 5. Apresentar os resultados
print("=" * 70)
print("NOTEBOOK 08 — IMPACTO GLOBAL DAS CORREÇÕES")
print("=" * 70)

for populacao in populacoes08:

    print(f"\nPOPULAÇÃO: {populacao}")

    tabela = metricas_globais08.loc[
        metricas_globais08["populacao"].eq(populacao),
        [
            "modelo", "cenario", "n",
            "rmse", "mae", "vies",
            "ganho_rmse_pct",
            "ganho_mae_pct",
            "ganho_vies_abs_pct"
        ]
    ]

    display(tabela.round(4))

# 6. Comparação direta do RMSE
print("\n" + "=" * 70)
print("COMPARAÇÃO DE RMSE — ANÁLISE PRINCIPAL")
print("=" * 70)

principal = metricas_globais08.loc[
    metricas_globais08["populacao"].eq("principal_6249h")
]

tabela_rmse08 = principal.pivot(
    index="modelo",
    columns="cenario",
    values="rmse"
)

tabela_rmse08["ganho_M1_pct"] = (
    100 * (tabela_rmse08["M0"] - tabela_rmse08["M1"])
    / tabela_rmse08["M0"]
)

tabela_rmse08["ganho_M2_pct"] = (
    100 * (tabela_rmse08["M0"] - tabela_rmse08["M2"])
    / tabela_rmse08["M0"]
)

display(tabela_rmse08.round(4))

# 7. Identificar o melhor cenário de cada modelo
melhores08 = (
    principal.sort_values(["modelo", "rmse"])
    .groupby("modelo", as_index=False)
    .first()
)

print("\nMELHOR CENÁRIO POR MODELO:")
display(
    melhores08[
        ["modelo", "cenario", "rmse", "mae", "vies"]
    ].round(4)
)

print("\nCÉLULA 06 CONCLUÍDA.")
print("Resultados calculados sem retreinamento.")

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 07 — AVALIAÇÃO TEMPORAL DAS CORREÇÕES
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# 1. Verificar dados
if "previsoes08" not in globals():
    raise RuntimeError("Execute a Célula 05 primeiro.")

MODELOS08 = ["A", "B", "C", "D"]
CENARIOS08 = ["M0", "M1", "M2"]

dados07_08 = previsoes08.copy()

# 2. Identificar períodos
nomes_periodos = {
    0: "P0 - Antes de março",
    1: "P1 - Março a julho",
    2: "P2 - Após julho"
}

dados07_08["periodo"] = (
    dados07_08["n_intervencoes"]
    .map(nomes_periodos)
)

dados07_08["mes"] = (
    dados07_08["timestamp"].dt.strftime("%Y-%m")
)

# 3. Função de avaliação
def metricas_periodo08(df, modelo, cenario):
    y = df["co2_metric_tons"].to_numpy(dtype=float)
    p = df[f"{modelo}_{cenario}"].to_numpy(dtype=float)
    erro = p - y

    return {
        "n": len(df),
        "rmse": float(np.sqrt(np.mean(erro**2))),
        "mae": float(np.mean(np.abs(erro))),
        "vies": float(np.mean(erro))
    }

# 4. Avaliação por período
linhas = []

for periodo, grupo in dados07_08.groupby(
    "periodo", sort=True
):
    for modelo in MODELOS08:
        for cenario in CENARIOS08:
            resultado = metricas_periodo08(
                grupo, modelo, cenario
            )

            linhas.append({
                "periodo": periodo,
                "modelo": modelo,
                "cenario": cenario,
                **resultado
            })

metricas_periodos08 = pd.DataFrame(linhas)

# 5. Calcular ganhos pareados em relação a M0
ref = metricas_periodos08.loc[
    metricas_periodos08["cenario"].eq("M0"),
    ["periodo", "modelo", "rmse", "mae"]
].rename(columns={
    "rmse": "rmse_original",
    "mae": "mae_original"
})

metricas_periodos08 = metricas_periodos08.merge(
    ref,
    on=["periodo", "modelo"],
    validate="many_to_one"
)

metricas_periodos08["ganho_rmse_pct"] = (
    100 * (
        metricas_periodos08["rmse_original"]
        - metricas_periodos08["rmse"]
    ) / metricas_periodos08["rmse_original"]
)

metricas_periodos08["ganho_mae_pct"] = (
    100 * (
        metricas_periodos08["mae_original"]
        - metricas_periodos08["mae"]
    ) / metricas_periodos08["mae_original"]
)

# 6. Exibir comparação temporal
print("=" * 70)
print("NOTEBOOK 08 — AVALIAÇÃO POR PERÍODO")
print("=" * 70)

for periodo in nomes_periodos.values():

    tabela = metricas_periodos08.loc[
        metricas_periodos08["periodo"].eq(periodo),
        [
            "modelo", "cenario", "n",
            "rmse", "mae", "vies",
            "ganho_rmse_pct", "ganho_mae_pct"
        ]
    ]

    print(f"\n{periodo}")
    display(tabela.round(4))

# 7. Avaliação mensal — cenário M2 versus M0
linhas_mensais = []

for mes, grupo in dados07_08.groupby("mes", sort=True):
    for modelo in MODELOS08:

        original = metricas_periodo08(
            grupo, modelo, "M0"
        )

        corrigido = metricas_periodo08(
            grupo, modelo, "M2"
        )

        linhas_mensais.append({
            "mes": mes,
            "modelo": modelo,
            "n": len(grupo),
            "rmse_M0": original["rmse"],
            "rmse_M2": corrigido["rmse"],
            "ganho_rmse_pct": (
                100 * (
                    original["rmse"] - corrigido["rmse"]
                ) / original["rmse"]
            ),
            "vies_M0": original["vies"],
            "vies_M2": corrigido["vies"]
        })

metricas_mensais08 = pd.DataFrame(linhas_mensais)

print("\n" + "=" * 70)
print("GANHO MENSAL DE RMSE — CORREÇÃO M2")
print("=" * 70)

tabela_mensal08 = metricas_mensais08.pivot(
    index="mes",
    columns="modelo",
    values="ganho_rmse_pct"
)

display(tabela_mensal08.round(2))

# 8. Identificar meses de piora
pioras08 = metricas_mensais08.loc[
    metricas_mensais08["ganho_rmse_pct"] < 0,
    ["mes", "modelo", "ganho_rmse_pct"]
]

print("\nMESES EM QUE M2 PIOROU O RMSE:")

if pioras08.empty:
    print("Nenhuma piora mensal identificada.")
else:
    display(pioras08.round(3))

# 9. Verificações finais
assert len(dados07_08) == 6249
assert metricas_periodos08["n"].gt(0).all()

print("\nCÉLULA 07 CONCLUÍDA.")
print("Análise temporal realizada sem retreinamento.")

## Inferência com dependência temporal — Célula 08

**Künsch (1989, pp. 1217–1218)** fundamenta reamostragem por blocos para observações temporalmente dependentes. O procedimento deste notebook agrega erros quadráticos/absolutos por dia e reamostra **blocos de sete dias**, conservando o pareamento dos cenários e modelos; o intervalo é construído por percentis empíricos de 1.000 reamostragens. A escolha de sete dias é **do projeto**, não um parâmetro imposto pelo artigo. Os intervalos são **condicionais** ao fator e às datas: não propagam a incerteza da calibração histórica nem a seleção de quebras (Jewell et al., 2022). Além disso, o fundamento de Künsch (1989, pp. 1217–1218) considera observações estacionárias; mudanças de regime e forte dependência residual podem comprometer a cobertura nominal de 95%. A sensibilidade aos comprimentos dos blocos é informativa, mas não substitui a verificação dessas condições.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 08 — BOOTSTRAP PAREADO EM BLOCOS DE 7 DIAS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

if "previsoes08" not in globals():
    raise RuntimeError("Execute a Célula 05 primeiro.")

MODELOS08 = ["A", "B", "C", "D"]
CENARIOS08 = ["M0", "M1", "M2"]

N_BOOT08 = 1000
BLOCO_DIAS08 = 7
SEMENTE08 = 20261119

dados_boot08 = previsoes08.sort_values("timestamp").copy()
dados_boot08["dia"] = dados_boot08["timestamp"].dt.normalize()

# 1. Agregar contribuições dos erros por dia.
# Isso mantém as observações horárias de cada dia juntas.
dados_boot08["n_horas"] = 1

colunas_agregacao = ["n_horas"]

for modelo in MODELOS08:
    for cenario in CENARIOS08:
        nome = f"{modelo}_{cenario}"
        erro = (
            dados_boot08[nome]
            - dados_boot08["co2_metric_tons"]
        )
        dados_boot08[f"sq_{nome}"] = erro**2
        dados_boot08[f"abs_{nome}"] = erro.abs()
        colunas_agregacao.extend([
            f"sq_{nome}", f"abs_{nome}"
        ])

diario08 = (
    dados_boot08.groupby("dia", sort=True)[colunas_agregacao]
    .sum()
)

# Incluir dias sem observações para preservar a distância
# cronológica real entre os blocos.
calendario08 = pd.date_range(
    diario08.index.min(),
    diario08.index.max(),
    freq="D"
)

diario08 = diario08.reindex(
    calendario08, fill_value=0.0
)

n_dias = len(diario08)

if n_dias < BLOCO_DIAS08:
    raise ValueError("Período insuficiente para bootstrap.")

# 2. Gerar uma única amostragem pareada,
# compartilhada por todos os modelos e cenários.
rng = np.random.default_rng(SEMENTE08)

n_blocos = int(np.ceil(n_dias / BLOCO_DIAS08))
max_inicio = n_dias - BLOCO_DIAS08

indices_boot = np.empty(
    (N_BOOT08, n_blocos * BLOCO_DIAS08),
    dtype=int
)

for b in range(N_BOOT08):
    inicios = rng.integers(
        0, max_inicio + 1, size=n_blocos
    )
    indices = np.concatenate([
        np.arange(i, i + BLOCO_DIAS08)
        for i in inicios
    ])
    indices_boot[b] = indices

indices_boot = indices_boot[:, :n_dias]

# 3. Calcular RMSE e MAE em cada reamostragem
n_por_dia = diario08["n_horas"].to_numpy()
n_boot = n_por_dia[indices_boot].sum(axis=1)

if np.any(n_boot == 0):
    raise AssertionError("Reamostragem sem observações.")

def metricas_boot08(modelo, cenario):
    nome = f"{modelo}_{cenario}"

    sq = diario08[f"sq_{nome}"].to_numpy()
    ab = diario08[f"abs_{nome}"].to_numpy()

    rmse = np.sqrt(
        sq[indices_boot].sum(axis=1) / n_boot
    )
    mae = (
        ab[indices_boot].sum(axis=1) / n_boot
    )
    return rmse, mae

# 4. Comparar ganhos pareados
resultados_boot = []

for modelo in MODELOS08:
    rmse0, mae0 = metricas_boot08(modelo, "M0")

    for cenario in ["M1", "M2"]:
        rmse1, mae1 = metricas_boot08(
            modelo, cenario
        )

        ganho_rmse = 100 * (rmse0 - rmse1) / rmse0
        ganho_mae = 100 * (mae0 - mae1) / mae0

        # Ganho observado na amostra original
        linha0 = metricas_globais08.loc[
            (metricas_globais08["populacao"] == "principal_6249h")
            & (metricas_globais08["modelo"] == modelo)
            & (metricas_globais08["cenario"] == cenario)
        ].iloc[0]

        resultados_boot.append({
            "modelo": modelo,
            "cenario": cenario,
            "ganho_rmse_pct": linha0["ganho_rmse_pct"],
            "ic95_rmse_inf": np.percentile(ganho_rmse, 2.5),
            "ic95_rmse_sup": np.percentile(ganho_rmse, 97.5),
            "ganho_mae_pct": linha0["ganho_mae_pct"],
            "ic95_mae_inf": np.percentile(ganho_mae, 2.5),
            "ic95_mae_sup": np.percentile(ganho_mae, 97.5),
            "frac_boot_sem_ganho_rmse": np.mean(ganho_rmse <= 0)
        })

bootstrap08 = pd.DataFrame(resultados_boot)

# 5. Apresentar resultados
print("=" * 70)
print("NOTEBOOK 08 — BOOTSTRAP PAREADO")
print("=" * 70)

print(f"Reamostragens: {N_BOOT08}")
print(f"Bloco: {BLOCO_DIAS08} dias")
print(f"Dias no calendário: {n_dias}")
print(f"Horas originais: {len(previsoes08)}")
print(f"Semente: {SEMENTE08}")

print("\nGANHOS E INTERVALOS DE CONFIANÇA DE 95%:")
display(bootstrap08.round(4))

print("\nCENÁRIOS COM IC95 DE RMSE INTEIRAMENTE POSITIVO:")
display(
    bootstrap08.assign(
        ic95_positivo=bootstrap08["ic95_rmse_inf"] > 0
    )[["modelo", "cenario", "ic95_positivo"]]
)

print("\nCÉLULA 08 CONCLUÍDA.")
print("Nenhum modelo foi retreinado.")

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 09 — VISUALIZAÇÃO DOS RESULTADOS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

necessarias = [
    "previsoes08",
    "metricas_globais08",
    "metricas_mensais08",
    "metricas_periodos08",
    "bootstrap08",
    "ROOT08"
]

ausentes = [v for v in necessarias if v not in globals()]

if ausentes:
    raise RuntimeError(
        f"Execute as células anteriores: {ausentes}"
    )

# 1. Pasta de resultados do Notebook 08
PASTA08 = ROOT08 / "outputs" / "notebook_08_correcao_carbonica"
FIGURAS08 = PASTA08 / "figuras"
FIGURAS08.mkdir(parents=True, exist_ok=True)

MODELOS08 = ["A", "B", "C", "D"]

# 2. Gráfico 1 — RMSE global
principal = metricas_globais08.loc[
    metricas_globais08["populacao"] == "principal_6249h"
]

tabela = principal.pivot(
    index="modelo", columns="cenario", values="rmse"
).loc[MODELOS08]

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(MODELOS08))
largura = 0.25

for i, cenario in enumerate(["M0", "M1", "M2"]):
    ax.bar(
        x + (i - 1) * largura,
        tabela[cenario],
        largura,
        label=cenario
    )

ax.set_xticks(x)
ax.set_xticklabels(MODELOS08)
ax.set_ylabel("RMSE (t CO₂/h)")
ax.set_xlabel("Modelo")
ax.set_title("RMSE global em 2025 — 6.249 horas")
ax.legend(title="Cenário")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()

fig.savefig(
    FIGURAS08 / "01_rmse_global.png",
    dpi=200, bbox_inches="tight"
)
plt.show()
plt.close(fig)

# 3. Gráfico 2 — Ganho mensal de RMSE com M2
mensal = metricas_mensais08.pivot(
    index="mes",
    columns="modelo",
    values="ganho_rmse_pct"
)

fig, ax = plt.subplots(figsize=(11, 5))

for modelo in MODELOS08:
    ax.plot(
        mensal.index,
        mensal[modelo],
        marker="o",
        label=modelo
    )

ax.axhline(0, linewidth=1, linestyle="--")
ax.set_ylabel("Redução do RMSE (%)")
ax.set_xlabel("Mês de 2025")
ax.set_title("Ganho mensal da correção M2")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Modelo")
ax.grid(alpha=0.2)
fig.tight_layout()

fig.savefig(
    FIGURAS08 / "02_ganho_mensal.png",
    dpi=200, bbox_inches="tight"
)
plt.show()
plt.close(fig)

# 4. Gráfico 3 — Viés por período, modelo A
periodos = [
    "P0 - Antes de março",
    "P1 - Março a julho",
    "P2 - Após julho"
]

vies_a = metricas_periodos08.loc[
    metricas_periodos08["modelo"] == "A"
].pivot(
    index="periodo",
    columns="cenario",
    values="vies"
).reindex(periodos)

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(periodos))
largura = 0.25

for i, cenario in enumerate(["M0", "M1", "M2"]):
    ax.bar(
        x + (i - 1) * largura,
        vies_a[cenario],
        largura,
        label=cenario
    )

ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([
    "Antes de março",
    "Março a julho",
    "Após julho"
])
ax.set_ylabel("Viés médio (t CO₂/h)")
ax.set_title("Viés por período — modelo A")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()

fig.savefig(
    FIGURAS08 / "03_vies_periodos_A.png",
    dpi=200, bbox_inches="tight"
)
plt.show()
plt.close(fig)

# 5. Gráfico 4 — Série temporal agregada semanalmente
# A agregação facilita a visualização sem ocultar
# os dados horários utilizados nas métricas.
serie = previsoes08[
    ["timestamp", "co2_metric_tons", "A_M0", "A_M2"]
].copy()

serie = serie.set_index("timestamp").sort_index()

semanal = serie.resample("7D").mean()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    semanal.index,
    semanal["co2_metric_tons"],
    label="Observado",
    linewidth=2
)

ax.plot(
    semanal.index,
    semanal["A_M0"],
    label="Modelo A — original",
    alpha=0.8
)

ax.plot(
    semanal.index,
    semanal["A_M2"],
    label="Modelo A — corrigido",
    alpha=0.8
)

for data in INTERVENCOES08:
    ax.axvline(
        data,
        linestyle="--",
        linewidth=1,
        alpha=0.7
    )

ax.set_title(
    "Emissões observadas e previstas — médias semanais"
)
ax.set_ylabel("Emissões médias (t CO₂/h)")
ax.set_xlabel("2025")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()

fig.savefig(
    FIGURAS08 / "04_series_semanais_A.png",
    dpi=200, bbox_inches="tight"
)
plt.show()
plt.close(fig)

print("=" * 65)
print("CÉLULA 09 CONCLUÍDA")
print("=" * 65)
print("Figuras salvas em:", FIGURAS08)

for arquivo in sorted(FIGURAS08.glob("*.png")):
    print("-", arquivo.name)

print("\nNenhum modelo foi retreinado.")
print("Nenhum arquivo original foi modificado.")

## Reprodutibilidade e artefatos — Células 09–10

A visualização resume comparações sobre as mesmas observações, e a exportação registra dados, tabelas, parâmetros e manifesto de arquivos. Esse procedimento implementa **rastreabilidade computacional própria**; não deriva de um artigo específico. As análises complementares posteriores à exportação original permanecem em seus artefatos próprios, sem modificar previsões ou resultados históricos.

In [ ]:
# ============================================================
# NOTEBOOK 08
# CÉLULA 10 — CONSOLIDAÇÃO E EXPORTAÇÃO
# ============================================================

import json
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# 1. Verificar dependências
necessarias = [
    "ROOT08", "RUN06", "ARQUIVO06", "ARQUIVO07",
    "previsoes08", "metricas_globais08",
    "metricas_periodos08", "metricas_mensais08",
    "bootstrap08", "FATOR08", "INTERVENCOES08"
]

ausentes = [v for v in necessarias if v not in globals()]

if ausentes:
    raise RuntimeError(
        f"Execute as células anteriores: {ausentes}"
    )

# 2. Estrutura de saída
PASTA08 = (
    ROOT08 / "outputs" / "notebook_08_correcao_carbonica"
)
DADOS08 = PASTA08 / "dados"
TABELAS08 = PASTA08 / "tabelas"
METADADOS08 = PASTA08 / "metadados"

for pasta in [DADOS08, TABELAS08, METADADOS08]:
    pasta.mkdir(parents=True, exist_ok=True)

# 3. Auditoria das previsões
assert len(previsoes08) == 6249
assert previsoes08["timestamp"].is_unique
assert int(previsoes08["hora_comum_07"].sum()) == 6002

for modelo in ["A", "B", "C", "D"]:
    assert np.array_equal(
        previsoes08[modelo].to_numpy(),
        previsoes08[f"{modelo}_M0"].to_numpy()
    )

# 4. Salvar previsões e tabelas
arquivos08 = {
    "previsoes_corrigidas": (
        DADOS08 / "previsoes_corrigidas_2025.parquet"
    ),
    "metricas_globais": (
        TABELAS08 / "metricas_globais.csv"
    ),
    "metricas_periodos": (
        TABELAS08 / "metricas_por_periodo.csv"
    ),
    "metricas_mensais": (
        TABELAS08 / "metricas_mensais.csv"
    ),
    "bootstrap": (
        TABELAS08 / "bootstrap_pareado_7dias.csv"
    )
}

previsoes08.to_parquet(
    arquivos08["previsoes_corrigidas"],
    index=False
)

metricas_globais08.to_csv(
    arquivos08["metricas_globais"], index=False
)

metricas_periodos08.to_csv(
    arquivos08["metricas_periodos"], index=False
)

metricas_mensais08.to_csv(
    arquivos08["metricas_mensais"], index=False
)

bootstrap08.to_csv(
    arquivos08["bootstrap"], index=False
)

# 5. Função para registrar integridade dos arquivos
def sha256_08(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

# 6. Parâmetros metodológicos
parametros08 = {
    "experimento": "Notebook 08 - Correcao carbonica",
    "unidade": "Iatan 2",
    "ano_avaliacao": 2025,
    "total_horas": int(len(previsoes08)),
    "horas_comuns_notebook07": int(
        previsoes08["hora_comum_07"].sum()
    ),
    "modelos": ["A", "B", "C", "D"],
    "cenarios": {
        "M0": "Previsao original",
        "M1": "Uma correcao apos primeira intervencao",
        "M2": "Correcao cumulativa por intervencao"
    },
    "fator_historico": float(FATOR08),
    "datas_intervencoes": [
        str(data) for data in INTERVENCOES08
    ],
    "bootstrap": {
        "reamostragens": int(N_BOOT08),
        "bloco_dias": int(BLOCO_DIAS08),
        "semente": int(SEMENTE08),
        "metodo": "Blocos moveis de dias, pareados",
        "intervalo_confianca": 0.95
    },
    "fontes": {
        "notebook06_run": (run_id or "20261119T013458Z_963fc189"),
        "mlflow_pai06": PAI_MLFLOW06,
        "previsoes06": str(ARQUIVO06),
        "previsoes07": str(ARQUIVO07),
        "fator07_csv": str(ARQUIVO_FATOR07),
        "relatorio07_final": str(ARQUIVO_RELATORIO07),
        "sha256_previsoes06": sha256_08(ARQUIVO06),
        "sha256_previsoes07": sha256_08(ARQUIVO07),
        "sha256_fator07": sha256_08(ARQUIVO_FATOR07),
        "sha256_relatorio07": sha256_08(ARQUIVO_RELATORIO07)
    },
    "auditoria_versao_corrigida07": {
        "horas_parciais_verificadas": int(mask_operacao_parcial.sum()),
        "max_diferenca_geracao_vs_mw_vezes_horas": maior_diferenca_geracao07,
        "max_diferenca_geracao_vs_carga_x_optime": maior_diferenca_identidade07,
        "arquivo_cems_original": str(ARQUIVO_RAW07),
        "fator_no_csv": FATOR07_CSV,
        "fator_no_relatorio": FATOR07_RELATORIO,
        "rmse07_m0_reproduzido": _rmse07_m0,
        "rmse07_m2_reproduzido": _rmse07_m2,
        "datas_no_relatorio07": [str(x) for x in _cortes_relatorio07]
    },
    "limitacoes": [
        "Datas de intervencao identificadas retrospectivamente",
        "Fator historico mantido fixo",
        "Nao houve retreinamento dos modelos",
        "Bootstrap condicional as intervencoes selecionadas",
        "Resultados nao constituem validacao prospectiva"
    ],
    "criado_utc": datetime.now(timezone.utc).isoformat()
}

arquivo_parametros = (
    METADADOS08 / "parametros_experimento.json"
)

with open(arquivo_parametros, "w", encoding="utf-8") as f:
    json.dump(
        parametros08, f,
        ensure_ascii=False,
        indent=2
    )

# 7. Criar manifesto de arquivos
manifesto = []

for nome, caminho in arquivos08.items():
    manifesto.append({
        "nome": nome,
        "arquivo": str(caminho.relative_to(PASTA08)),
        "bytes": caminho.stat().st_size,
        "sha256": sha256_08(caminho)
    })

manifesto.append({
    "nome": "parametros",
    "arquivo": str(
        arquivo_parametros.relative_to(PASTA08)
    ),
    "bytes": arquivo_parametros.stat().st_size,
    "sha256": sha256_08(arquivo_parametros)
})

manifesto08 = pd.DataFrame(manifesto)

arquivo_manifesto = (
    METADADOS08 / "manifesto_arquivos.csv"
)

manifesto08.to_csv(arquivo_manifesto, index=False)

# 8. Verificar leitura dos arquivos exportados
assert len(pd.read_parquet(
    arquivos08["previsoes_corrigidas"]
)) == 6249

assert len(pd.read_csv(
    arquivos08["metricas_globais"]
)) == 24

assert len(pd.read_csv(
    arquivos08["bootstrap"]
)) == 8

# 9. Resumo final
print("=" * 70)
print("NOTEBOOK 08 — EXPERIMENTO CONSOLIDADO")
print("=" * 70)

print("\nDiretório:", PASTA08)
print("Horas avaliadas:", len(previsoes08))
print("Modelos: A, B, C, D")
print("Cenários: M0, M1, M2")
print("Fator:", round(FATOR08, 10))

print("\nARQUIVOS EXPORTADOS:")
display(
    manifesto08[["nome", "arquivo", "bytes"]]
)

print("\nRESUMO FINAL — CENÁRIO M2:")

resumo_final08 = metricas_globais08.loc[
    (metricas_globais08["populacao"] == "principal_6249h")
    & (metricas_globais08["cenario"] == "M2"),
    [
        "modelo", "rmse", "mae", "vies",
        "ganho_rmse_pct", "ganho_mae_pct"
    ]
]

display(resumo_final08.round(4))

print("\nNOTEBOOK 08 FINALIZADO.")
print("Previsões e resultados exportados.")
print("Arquivos originais preservados.")

## Diagnósticos complementares — cobertura e resíduos

O conjunto contém **6.249 observações de um calendário de 8.760 horas (71,34% de cobertura anual)**; em outubro a cobertura observada é de **28,36%**. A ausência de observações não deve ser tratada como emissão zero nem como série contínua. As tabelas abaixo avaliam a distribuição das horas disponíveis, o viés e a dependência temporal dos resíduos. A autocorrelação calculada sobre pares de horas consecutivas é apenas descritiva.

---

**Base metodológica — EPA (2022, p. 8, *Hourly Data*); Hyndman e Koehler (2006, §2.1, p. 682).** Cobertura, qualidade dos registros observados, resíduos e viés são componentes distintos do diagnóstico de precisão. Uma amostra de 6.249 horas **não representa necessariamente uma amostra aleatória** das 8.760 horas do ano; lacunas podem estar associadas à operação. Autocorrelação descritiva de resíduos não constitui, por si só, um teste de especificação ou validação prospectiva.

In [ ]:
# ============================================================
# CÉLULA 11 — COBERTURA E DIAGNÓSTICO DOS RESÍDUOS
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import display

assert "previsoes08" in globals(), "Execute as células 01–10 antes."
df11 = previsoes08.sort_values("timestamp").copy()
assert df11["timestamp"].is_unique
assert len(df11) == 6249

todas_horas11 = pd.date_range("2025-01-01", "2025-12-31 23:00", freq="h")
observadas11 = pd.DatetimeIndex(df11["timestamp"])
assert observadas11.isin(todas_horas11).all(), "Há timestamps fora de 2025 ou desalinhados à hora."

cobertura11 = (pd.DataFrame({"timestamp": todas_horas11})
    .assign(mes=lambda x: x["timestamp"].dt.strftime("%Y-%m"),
            disponivel=lambda x: x["timestamp"].isin(observadas11))
    .groupby("mes", as_index=False)
    .agg(horas_calendario=("disponivel", "size"), horas_disponiveis=("disponivel", "sum")))
cobertura11["cobertura_pct"] = 100*cobertura11["horas_disponiveis"]/cobertura11["horas_calendario"]

linhas11 = []
pares_consecutivos11 = df11["timestamp"].diff().eq(pd.Timedelta(hours=1)).to_numpy()
for modelo in ["A", "B", "C", "D"]:
    for cenario in ["M0", "M1", "M2"]:
        e = (df11[f"{modelo}_{cenario}"]-df11["co2_metric_tons"]).to_numpy(dtype=float)
        pares_atual = e[1:][pares_consecutivos11[1:]]
        pares_anterior = e[:-1][pares_consecutivos11[1:]]
        acf1 = (float(np.corrcoef(pares_anterior, pares_atual)[0,1])
                if len(pares_atual)>2 and np.std(pares_atual)>0 and np.std(pares_anterior)>0 else np.nan)
        linhas11.append({"modelo":modelo,"cenario":cenario,"n":len(e),
            "vies":float(e.mean()),"mediana_erro":float(np.median(e)),
            "p05_erro":float(np.percentile(e,5)),"p95_erro":float(np.percentile(e,95)),
            "acf_lag1_horas_consecutivas":acf1,"pares_consecutivos":len(pares_atual)})
residuos11 = pd.DataFrame(linhas11)

print("COBERTURA ANUAL:",len(df11),"/",len(todas_horas11),
      f"({100*len(df11)/len(todas_horas11):.2f}%)")
display(cobertura11.round(2))
print("RESÍDUOS (previsão − observado):")
display(residuos11.round(4))

# Salvar tabelas complementares sem sobrescrever resultados originais.
PASTA08 = ROOT08 / "outputs" / "notebook_08_correcao_carbonica"
TABELAS_COMPLEMENTARES08 = PASTA08 / "tabelas_complementares"
TABELAS_COMPLEMENTARES08.mkdir(parents=True,exist_ok=True)
cobertura11.to_csv(TABELAS_COMPLEMENTARES08/"cobertura_mensal_2025.csv",index=False)
residuos11.to_csv(TABELAS_COMPLEMENTARES08/"diagnostico_residuos.csv",index=False)
print("Diagnósticos salvos em:",TABELAS_COMPLEMENTARES08)

## Sensibilidade dos intervalos bootstrap ao comprimento do bloco

O procedimento original utilizou blocos móveis de sete dias. A análise abaixo repete a comparação **pareada** com blocos de **3, 7, 14 e 21 dias**, mantendo as observações de cada dia juntas e incluindo dias sem observações na linha temporal. O objetivo é verificar se a conclusão de ganho positivo depende fortemente da escolha do bloco. **Nesta execução, os 32 intervalos de confiança calculados (modelos A–D, cenários M1 e M2, e quatro comprimentos de bloco) tiveram limite inferior positivo**, sem que isso elimine as ressalvas sobre seleção retrospectiva.

**Nota:** os intervalos são percentis bootstrap, não testes causais, e **não incorporam** incerteza sobre a seleção retrospectiva das datas ou do fator. São comparações condicionais aos dados e cenários fixados.

---

**Fundamento — Künsch (1989, pp. 1217–1218).** O bootstrap em blocos procura preservar parte da dependência existente entre observações sucessivas; isso é particularmente importante quando o erro horário apresenta persistência. A comparação M0/M2 continua **pareada**, e a sensibilidade a 3, 7, 14 e 21 dias avalia uma escolha analítica, sem provar a validade universal dos intervalos. **Jewell et al. (2022, §1, pp. 1082–1084)** motivam a ressalva de que o procedimento não corrige a incerteza de ter identificado previamente os cortes na própria série de avaliação.

In [ ]:
# ============================================================
# CÉLULA 12 — SENSIBILIDADE DO BOOTSTRAP (3, 7, 14, 21 DIAS)
# ============================================================
import numpy as np
import pandas as pd
from IPython.display import display

N_BOOT_SENS08 = 1000
SEMENTE_SENS08 = 20261119
BLOCOS_SENS08 = [3, 7, 14, 21]

base12 = previsoes08.sort_values("timestamp").copy()
base12["dia"] = base12["timestamp"].dt.normalize()
base12["n_horas"] = 1
cols12 = ["n_horas"]
for m in ["A", "B", "C", "D"]:
    for c in ["M0", "M1", "M2"]:
        erro = base12[f"{m}_{c}"]-base12["co2_metric_tons"]
        nome = f"sq_{m}_{c}"
        base12[nome] = erro.pow(2)
        cols12.append(nome)

dias12 = base12.groupby("dia",sort=True)[cols12].sum()
cal12 = pd.date_range("2025-01-01", "2025-12-31",freq="D")
dias12 = dias12.reindex(cal12,fill_value=0.0)
n12 = dias12["n_horas"].to_numpy()
N_DIAS12 = len(dias12)

linhas12 = []
for bloco in BLOCOS_SENS08:
    rng = np.random.default_rng(SEMENTE_SENS08 + bloco)
    n_blocos = int(np.ceil(N_DIAS12/bloco))
    inicios = rng.integers(0,N_DIAS12-bloco+1,size=(N_BOOT_SENS08,n_blocos))
    indices = (inicios[:,:,None] + np.arange(bloco)[None,None,:]).reshape(N_BOOT_SENS08,-1)[:,:N_DIAS12]
    contagens = n12[indices].sum(axis=1)
    assert (contagens>0).all()
    for modelo in ["A", "B", "C", "D"]:
        sq0 = dias12[f"sq_{modelo}_M0"].to_numpy()
        rmse0 = np.sqrt(sq0[indices].sum(axis=1)/contagens)
        for cenario in ["M1", "M2"]:
            sq1 = dias12[f"sq_{modelo}_{cenario}"].to_numpy()
            rmse1 = np.sqrt(sq1[indices].sum(axis=1)/contagens)
            ganho = 100*(rmse0-rmse1)/rmse0
            linhas12.append({"bloco_dias":bloco,"modelo":modelo,"cenario":cenario,
                "ic95_inf":float(np.percentile(ganho,2.5)),
                "mediana_boot":float(np.median(ganho)),
                "ic95_sup":float(np.percentile(ganho,97.5)),
                "frac_sem_ganho":float(np.mean(ganho<=0)),
                "ic95_inteiramente_positivo":bool(np.percentile(ganho,2.5)>0)})

sensibilidade12 = pd.DataFrame(linhas12)
display(sensibilidade12.round(4))
print("Cenários com IC95 inteiramente positivo:",
      int(sensibilidade12["ic95_inteiramente_positivo"].sum()),
      "/",len(sensibilidade12))
sensibilidade12.to_csv(TABELAS_COMPLEMENTARES08/"sensibilidade_bootstrap_blocos.csv",index=False)
print("Tabela salva em:",TABELAS_COMPLEMENTARES08)

## Visualização complementar — respeitando lacunas de observação

A figura original de médias semanais é útil, mas uma média de uma semana com poucas observações pode ser interpretada indevidamente como representativa de todos os sete dias. A figura abaixo apresenta **médias diárias apenas para dias com observações**, interrompe linhas nos dias sem dados e indica a cobertura diária. As métricas de avaliação permanecem estritamente horárias.

---

**Interpretação e fonte — EPA (2022, p. 8, *Hourly Data*); Hyndman e Koehler (2006, §2.1, p. 682).** A média diária de emissões corresponde apenas às horas disponíveis em cada dia. Preservar os dias sem registros como lacunas evita criar visualmente continuidade que não existe nos dados. Os gráficos são **diagnósticos descritivos**; RMSE e MAE permanecem calculados sobre pares horários de previsão e observação.

In [ ]:
# ============================================================
# CÉLULA 13 — SÉRIES DIÁRIAS SEM CONECTAR LACUNAS
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

serie13 = previsoes08[["timestamp","co2_metric_tons","A_M0","A_M2"]].copy()
serie13 = serie13.set_index("timestamp").sort_index()
medias13 = serie13.resample("D").mean()
contagens13 = serie13["co2_metric_tons"].resample("D").count()

fig, (ax, ax2) = plt.subplots(2,1,figsize=(13,7),sharex=True,
                            gridspec_kw={"height_ratios":[3,1]})
for coluna, rotulo in [("co2_metric_tons","Observado"),
                       ("A_M0","A — original"),("A_M2","A — M2")]:
    ax.plot(medias13.index,medias13[coluna],label=rotulo,linewidth=1.2)
for data in INTERVENCOES08:
    ax.axvline(data,linestyle="--",alpha=0.7)
    ax2.axvline(data,linestyle="--",alpha=0.7)
ax.set_title("Emissões médias diárias — linhas interrompidas nos dias sem observações")
ax.set_ylabel("t CO₂/h (média das horas disponíveis)")
ax.legend()
ax.grid(alpha=0.2)
ax2.bar(contagens13.index,contagens13.values,width=0.9)
ax2.set_ylim(0,24)
ax2.set_ylabel("Horas/dia")
ax2.set_xlabel("2025")
ax2.grid(axis="y",alpha=0.2)
fig.tight_layout()
figuras13 = PASTA08 / "figuras"
figuras13.mkdir(parents=True,exist_ok=True)
saida13 = figuras13 / "05_series_diarias_com_cobertura.png"
fig.savefig(saida13,dpi=200,bbox_inches="tight")
plt.show()
plt.close(fig)
print("Figura complementar salva:",saida13)

In [ ]:
# ============================================================
# CÉLULA 14 — SÍNTESE AUTOMÁTICA DA REEXECUÇÃO
# Os números são gerados a partir das métricas da execução ATUAL.
# ============================================================

from IPython.display import display
import numpy as np
import pandas as pd

necessarias = ["metricas_globais08", "bootstrap08", "FATOR08", "previsoes08"]
ausentes = [x for x in necessarias if x not in globals()]
if ausentes:
    raise RuntimeError(f"Execute as células anteriores. Ausentes: {ausentes}")

resumo_final_dinamico08 = (
    metricas_globais08.loc[
        metricas_globais08["populacao"].eq("principal_6249h")
        & metricas_globais08["cenario"].isin(["M0", "M2"]),
        ["modelo", "cenario", "n", "rmse", "mae", "vies", "ganho_rmse_pct", "ganho_mae_pct"]
    ]
    .sort_values(["modelo", "cenario"])
    .reset_index(drop=True)
)

intervalos_m2_08 = bootstrap08.loc[
    bootstrap08["cenario"].eq("M2"),
    ["modelo", "ic95_rmse_inf", "ic95_rmse_sup", "frac_boot_sem_ganho_rmse"]
].copy()
intervalos_m2_08["ic95_positivo"] = intervalos_m2_08["ic95_rmse_inf"].gt(0)

print("=" * 72)
print("NOTEBOOK 08 — CONCLUSÕES NUMÉRICAS DA VERSÃO REEXECUTADA")
print("=" * 72)
print(f"Fator histórico recuperado do 07 corrigido: F = {FATOR08:.10f}")
print(f"Horas do conjunto principal (Notebook 06): {len(previsoes08)}")
print(f"Horas em comum com o Notebook 07: {previsoes08['hora_comum_07'].sum():.0f}")
print("\nRMSE, MAE, viés e ganhos por modelo (mesma população por cenário):")
display(resumo_final_dinamico08.round(4))

print("\nIntervalos bootstrap pareados de 95% — ganho de RMSE M2 versus M0:")
display(intervalos_m2_08.round(4))

melhor_m2_08 = (
    resumo_final_dinamico08.loc[resumo_final_dinamico08["cenario"].eq("M2")]
    .sort_values("rmse")
    .iloc[0]
)
print(
    "\nMenor RMSE absoluto entre os modelos A–D no cenário M2:",
    f"{melhor_m2_08['modelo']} ({melhor_m2_08['rmse']:.4f} tCO₂/h)"
)
print(
    "IC95 positivo do ganho M2 x M0 para todos os modelos:",
    "SIM" if intervalos_m2_08["ic95_positivo"].all() else "NÃO"
)
print("\nINTERPRETAÇÃO: análise condicional retrospectiva; não validação prospectiva.")
print("CÉLULA 14 CONCLUÍDA")


from iatan.carbon_tracking import log_notebook08
print("Registro no MLflow:", log_notebook08(ROOT08, PASTA08, ARQUIVO06, ARQUIVO_FATOR07, ARQUIVO_DATAS07, PAI_MLFLOW06, PASTA07_FONTE))

## Discussão

A síntese acima compara M0, M1 e M2 sobre as mesmas previsões da rodada anual de 2025. A comparação com as 6.002 horas do 07 é pareada por timestamp e pela massa observada. O registro separado das horas em cada período distingue cobertura, elegibilidade do 07 e aplicação de F. Os intervalos percentis do bootstrap mantêm datas e fator fixos e não propagam a incerteza da seleção com CO₂ de 2025. O 08 não inclui o cenário RH nem o período reservado de 2026.

## Referências científicas e documentação metodológica

Os trabalhos abaixo fundamentam os métodos e as limitações **efetivamente relacionados ao Notebook 08**. As definições completas da calibração histórica do fator \(F\) e da identificação das datas constam do **Notebook 07**; tais procedimentos **não são repetidos nem validados prospectivamente** aqui.

1. **U.S. ENVIRONMENTAL PROTECTION AGENCY — EPA (2022, p. 8, *Hourly Data*).** *CAMD's Power Sector Emissions Data Guide*. U.S. EPA. [Acesso à documentação](https://www.epa.gov/power-sector/power-sector-emission-data-guide). **Uso:** interpretação das variáveis horárias, cobertura, ajuste por tempo operacional e limites dos metadados de substituição no CAMPD em comparação com FTP e FACT/MODC.
2. **AKPAN, P. U.; FULS, W. F. (2021).** Cycling of coal fired power plants: A generic CO₂ emissions factor model for predicting CO₂ emissions. *Energy*, 214, 119026. DOI: [10.1016/j.energy.2020.119026](https://doi.org/10.1016/j.energy.2020.119026). **Uso:** motivação física para investigar intensidade de emissões variável com a operação; não comprova manutenção nem efeito causal nas datas selecionadas.
3. **CHEN, T.; GUESTRIN, C. (2016).** XGBoost: A scalable tree boosting system. In: *Proceedings of KDD '16*, pp. 785–794. DOI: [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). **Uso:** base teórica dos componentes XGBoost previamente treinados no Notebook 06.
4. **RABINER, L. R. (1989).** A tutorial on hidden Markov models and selected applications in speech recognition. *Proceedings of the IEEE*, 77(2), 257–286. DOI: [10.1109/5.18626](https://doi.org/10.1109/5.18626). **Uso:** fundamento conceitual dos modelos ocultos de Markov (HMM) utilizados na construção anterior do modelo C; **nenhum HMM é ajustado aqui**.
5. **VAN DER AALST, W. M. P. (2022).** Process Mining: A 360 Degree Overview. *Process Mining Handbook*, pp. 3–34. [DOI](https://doi.org/10.1007/978-3-031-08848-3_1). Uso: conceitos de log e grafo de sucessão direta, pp. 10 e 12; a construção dos atributos é descrita no 06.
6. **GAMA, J.; ŽLIOBAITĖ, I.; BIFET, A.; PECHENIZKIY, M.; BOUCHACHIA, A. (2014).** A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), artigo 44. DOI: [10.1145/2523813](https://doi.org/10.1145/2523813). **Uso:** motivação para corrigir previsões sob mudanças da relação entre geração e emissões; não garante transferência do mesmo fator entre modelos.
7. **CERQUEIRA, V.; TORGO, L.; MOZETIČ, I. (2020).** Evaluating time series forecasting models: An empirical study on performance estimation methods. *Machine Learning*, 109, 1997–2028. DOI: [10.1007/s10994-020-05910-7](https://doi.org/10.1007/s10994-020-05910-7). **Uso:** avaliação fora da amostra e respeito à ordem temporal; o presente experimento continua **retrospectivo** quanto à seleção das datas.
8. **HYNDMAN, R. J.; KOEHLER, A. B. (2006).** Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688. DOI: [10.1016/j.ijforecast.2006.03.001](https://doi.org/10.1016/j.ijforecast.2006.03.001). **Uso:** métricas de acurácia preditiva e interpretação de RMSE/MAE e medidas de erro.
9. **KÜNSCH, H. R. (1989).** The jackknife and the bootstrap for general stationary observations. *The Annals of Statistics*, 17(3), 1217–1241. DOI: [10.1214/aos/1176347265](https://doi.org/10.1214/aos/1176347265). **Uso:** motivação do bootstrap em blocos diante de dependência temporal; comprimentos 3/7/14/21 dias são decisões de sensibilidade do estudo.
10. **JEWELL, S.; FEARNHEAD, P.; WITTEN, D. (2022).** Testing for a change in mean after changepoint detection. *Journal of the Royal Statistical Society: Series B*, 84(4), 1082–1104. DOI: [10.1111/rssb.12501](https://doi.org/10.1111/rssb.12501). **Uso:** ressalva sobre inferência após seleção dos cortes; **a inferência seletiva proposta pelos autores não foi executada** neste notebook.

**Fonte de dados primários — citação recomendada pela EPA (2022, p. 8):** UNITED STATES ENVIRONMENTAL PROTECTION AGENCY (EPA). *Power Sector Emissions Data*. Washington, DC: Office of Atmospheric Programs, Clean Air Markets Division. Disponível em: [EPA Clean Air Markets Program Data — CAMPD](https://campd.epa.gov). Esta é a **fonte empírica**, distinta da referência ao guia técnico, e não corresponde a um novo artigo científico.

**Referência cruzada indispensável:** o Notebook 07 registra o procedimento de estimação do fator histórico a partir de 2017, o refinamento das datas de 2025 e as sensibilidades correspondentes. O Notebook 08 testa a **aplicabilidade retrospectiva** do fator às previsões **já treinadas** no Notebook 06. São perguntas científicas distintas, com amostras diferentes.